# Complexidade e busca

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Prever, pelos limites da entrada, se uma solução cabe no limite de tempo
- Trocar um laço aninhado por um conjunto de hash, ordenação ou dois ponteiros
- Achar fronteiras em dados ordenados com busca binária e contar os valores de um intervalo
- Fazer busca binária na resposta quando testar um candidato é fácil, mas achá-lo direto é difícil

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## Em resumo

Limites de tempo transformam o Big O num orçamento. Um computador faz mais ou menos **10⁸ passos simples por segundo**, então leia os limites da entrada antes de escrever código: com n até 200.000, um laço sobre todos os pares dá 4 × 10¹⁰ passos (minutos), enquanto O(n log n) dá uns 3,5 milhões (milissegundos). Como regra prática, n ≤ 5.000 permite O(n²), e n ≤ 500.000 pede O(n log n) ou melhor.

**Rode e olhe as colunas:** quando n dobra, um laço faz o dobro do trabalho, e um laço dentro de outro faz quatro vezes mais.

In [ ]:
for (int n = 1000; n <= 16000; n *= 2) {
    long um = 0;
    long aninhado = 0;
    for (int i = 0; i < n; i++) {
        um++;
    }
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            aninhado++;
        }
    }
    IO.println(String.format("n = %,6d   um laço: %,7d   laço dentro de laço: %,12d", n, um, aninhado));
}

### ✏️ Exercício: Conte as metades

Quantas vezes dá para dividir n por 2 (divisão inteira) até chegar a 1? Esse número é cerca de log₂ n, os passos de que a busca binária precisa.

In [ ]:
int halvings(long n) {
    // TODO: conte quantas vezes n é dividido por 2 até virar 1
    return 0;
}

void main() {
    check("n = 1", halvings(1), 0);
    check("n = 8", halvings(8), 3);
    check("n = 1000", halvings(1000), 9);
    check("um bilhão", halvings(1_000_000_000L), 29);
}

<details>
<summary>Ver uma resposta</summary>

```java
int halvings(long n) {
    int count = 0;
    while (n > 1) {
        n /= 2;
        count++;
    }
    return count;
}

void main() {
    check("n = 1", halvings(1), 0);
    check("n = 8", halvings(8), 3);
    check("n = 1000", halvings(1000), 9);
    check("um bilhão", halvings(1_000_000_000L), 29);
}
```

</details>

A maioria das correções troca um **laço aninhado** por uma ferramenta melhor:

- **Um conjunto de hash** responde "já vi isto antes?" em tempo constante, em média.
- **Ordenar** coloca valores iguais ou próximos lado a lado, e uma passada compara vizinhos.
- **Dois ponteiros** percorrem um array ordenado pelas duas pontas, ou como início e fim de uma janela, cada um andando num só sentido.

**Cronometre os dois jeitos** de achar ids repetidos. Depois troque `n` para 40.000: qual tempo dobra, e qual fica quatro vezes maior?

In [ ]:
int n = 20_000;
int[] ids = new Random(1).ints(n, 0, 1_000_000_000).toArray();

long inicio = System.nanoTime();
int repetidosLento = 0; // ids já vistos antes na lista
for (int i = 0; i < n; i++) {
    for (int j = 0; j < i; j++) {
        if (ids[i] == ids[j]) {
            repetidosLento++;
            break;
        }
    }
}
long meio = System.nanoTime();
Set<Integer> vistos = new HashSet<>();
int repetidosRapido = 0;
for (int id : ids) {
    if (!vistos.add(id)) {
        repetidosRapido++;
    }
}
long fim = System.nanoTime();

IO.println("laço dentro de laço: " + (meio - inicio) / 1_000_000 + " ms, repetidos: " + repetidosLento);
IO.println("HashSet:             " + (fim - meio) / 1_000_000 + " ms, repetidos: " + repetidosRapido);

### ✏️ Exercício: Valores em comum com dois ponteiros

Os dois arrays estão ordenados e não têm repetidos. Conte os valores em comum numa passada: avance o ponteiro do menor valor, e conte quando os dois forem iguais.

In [ ]:
int commonCount(int[] a, int[] b) {
    // TODO: percorra os dois arrays com i e j
    return 0;
}

void main() {
    check("dois em comum", commonCount(new int[] {1, 3, 5, 7}, new int[] {3, 4, 7}), 2);
    check("nenhum", commonCount(new int[] {1, 2}, new int[] {3, 4}), 0);
    check("um vazio", commonCount(new int[] {}, new int[] {1}), 0);
    check("todos", commonCount(new int[] {2, 4, 6}, new int[] {2, 4, 6}), 3);
}

<details>
<summary>Ver uma resposta</summary>

```java
int commonCount(int[] a, int[] b) {
    int i = 0, j = 0, common = 0;
    while (i < a.length && j < b.length) {
        if (a[i] == b[j]) {
            common++;
            i++;
            j++;
        } else if (a[i] < b[j]) {
            i++;
        } else {
            j++;
        }
    }
    return common;
}

void main() {
    check("dois em comum", commonCount(new int[] {1, 3, 5, 7}, new int[] {3, 4, 7}), 2);
    check("nenhum", commonCount(new int[] {1, 2}, new int[] {3, 4}), 0);
    check("um vazio", commonCount(new int[] {}, new int[] {1}), 0);
    check("todos", commonCount(new int[] {2, 4, 6}, new int[] {2, 4, 6}), 3);
}
```

</details>

A **busca binária** faz mais que achar um valor. Em dados ordenados ela acha **fronteiras**, como o primeiro índice com valor ≥ x, e duas fronteiras contam tudo o que está num intervalo:

![O primeiro valor ≥ 4 e o primeiro valor > 4 num array ordenado](boundaries.pt-br.svg)

#### 🎞️ Busca binária, passo a passo

![Busca binária do 7 num array ordenado: cada teste corta o intervalo pela metade até achar o 7 no índice 4](binary-search-steps.pt-br.svg)

A próxima célula começa com `// shared`, então ela define `lowerBound` para as células comuns abaixo. Rode-a, depois a que a usa. (As células de desafio rodam sozinhas, como o Submit: copie para elas o que precisar.)

In [ ]:
// shared: as células abaixo podem chamar lowerBound
/** O primeiro índice com valor >= x, ou a.length se não houver. */
int lowerBound(int[] a, long x) {
    int lo = 0;
    int hi = a.length;
    while (lo < hi) {
        int mid = (lo + hi) >>> 1;
        if (a[mid] < x) {
            lo = mid + 1;
        } else {
            hi = mid;
        }
    }
    return lo;
}

In [ ]:
int[] a = {2, 4, 4, 4, 7, 9};
IO.println("primeiro >= 4: índice " + lowerBound(a, 4));
IO.println("primeiro > 4:  índice " + lowerBound(a, 5)); // "primeiro > 4" é "primeiro >= 5"
IO.println("quantos 4: " + (lowerBound(a, 5) - lowerBound(a, 4)));
IO.println("primeiro >= 10: índice " + lowerBound(a, 10) + " (depois do fim)");
// Experimente: quantos valores estão entre 3 e 8, inclusive?

Ela também busca uma **resposta**: quando "c funciona?" é não até certo valor e sim dali em diante, a busca binária acha o primeiro sim em uns 30 testes, mesmo entre 1 e 10⁹.

**Busca binária na resposta:** o menor x com x × x ≥ alvo. Experimente outros alvos.

In [ ]:
long alvo = 2_000_000_000_000L; // qualquer alvo de 0 a 9_000_000_000_000_000_000L
long lo = 0;
long hi = 3_037_000_499L; // o maior x cujo x * x cabe num long
int testes = 0;
while (lo < hi) {
    long mid = (lo + hi) / 2;
    testes++;
    if (mid * mid >= alvo) {
        hi = mid;
    } else {
        lo = mid + 1;
    }
}
IO.println("menor x: " + lo + ", achado em " + testes + " testes");

O erro mais comum é errar por um: decida se `hi` é o último índice ou um depois dele, e mantenha essa escolha em todo lugar.

## Leituras

- [Time Complexity](https://usaco.guide/bronze/time-comp) (USACO Guide, 15 min): A tabela em "Common Complexities and Constraints": ela transforma um limite da entrada na complexidade mais lenta que ainda cabe. Os exemplos podem aparecer em Java: escolha a linguagem nas configurações do site.
- [Binary Search](https://usaco.guide/silver/binary-search) (USACO Guide, 20 min): Busca binária numa função de sim ou não (o primeiro x em que f(x) é verdadeira) e os três erros comuns: errar por um, limites negativos e estouro. O primeiro índice com valor ≥ x é a mesma busca, com f(i) = "a[i] ≥ x". Pule as listas de problemas.
- [Two Pointers](https://usaco.guide/silver/two-pointers) (USACO Guide, 12 min): "Sum of Two Values" (ponteiros vindo das duas pontas de um array ordenado) e "Sliding Window" (um início e um fim que só andam para a direita, numa animação). Pule as listas de problemas.

## Verifique

1. Com n ≤ 100.000, uma solução O(n²) cabe num limite de um segundo?
2. Em `2 4 4 4 7 9`, qual é o primeiro índice com valor ≥ 5?
3. Por que dá para fazer busca binária na menor capacidade de caminhão que funciona, se nenhum array guarda as capacidades?

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Ingressos repetidos** · Fácil
- **Janela de log** · Médio
- **Capacidade do caminhão** · Médio
- **Limitador de requisições** · Difícil